# Exploring the Jewish Concert Archive JSON

This notebook is a guided tour of `jewish_concert_archive.json` — a hand-curated catalog of concerts, revues, and theatrical performances documented in the digitized papers of violinist/orchestra leader Majer "Ivan" Pietruschka (see the archive's own `archive_overview` field, loaded below, for the full story).

The goals for this workshop session are to:

1. Understand how the JSON is actually structured (which is *not* a single flat table).
2. Harvest four things from it: **locations**, **dates**, **event types**, and **persons involved (with their roles)**.
3. Build a tidy `pandas` table and make some exploratory bar charts and scatter plots with **Plotly Express**.
4. Prepare (but not yet build) the data for a later **bi-nodal Person–Place network**, which we'll construct in a follow-up notebook using `networkx` and `pyvis`.

Throughout, the emphasis is on showing the *technique* for harvesting messy, real-world nested JSON — not just the final numbers. Several of the choices below (how we bucket event types, how we shorten location names) are simple heuristics meant to be inspected and adjusted, not treated as ground truth.

In [4]:
import json
import re
from collections import Counter

import pandas as pd
import plotly.express as px


## Loading the archive

At the top level, the JSON is a dictionary with just two keys:

- `archive_overview` — a single prose string describing the collection as a whole.
- `concerts` — a list of dictionaries, one per cataloged event (despite the key's name, this includes concerts, revues, plays, and even a few standalone songs — see below).

There is no schema file enforcing what's inside each concert dictionary — it was built by a human cataloger working from archival documents, so the fields present vary from record to record depending on what that particular document actually contained. Handling that variability *is* the main exercise in this notebook.

In [5]:
with open("jewish_concert_archive.json") as f:
    archive = json.load(f)

concerts = archive["concerts"]

print(archive["archive_overview"])
print(f"\n{len(concerts)} cataloged events.")


This JSON catalogs concert, revue, and theatrical performance programs found in a digitized collection of personal papers belonging to (or collected by) Majer 'Ivan' Pietruschka, a Polish-born violinist and orchestra leader. Pietruschka played in the Warsaw and Lodz Symphony Orchestras and conducted a silent-film orchestra in Berlin (1923-1932) before Nazi restrictions on Jewish musicians drove him to England in 1939. In 1940 the British Government transported him and other refugees to Australia aboard the HMT/HMAT Dunera (attacked three times by German U-boats en route), where he was interned as an 'enemy alien,' first apparently at Hay Camp (NSW) and later at Tatura (Victoria). After release, he joined the Australian Army's 8th Employment Company, entertaining troops with an orchestra he formed, and after the war played with the 3DB Orchestra and, from 1952, the Melbourne Symphony Orchestra. He married pianist-composer Phyllis Batchelor in 1946 and settled in Heidelberg, Victoria. Ma

## Anatomy of a single record

Let's look at one full record to see what a concert dictionary actually contains, then list just its keys for something easier to scan.

In [6]:
example = concerts[0]
print(json.dumps(example, indent=2)[:2000], "...\n")
print("Keys in this record:", sorted(example.keys()))


{
  "title": "Snowhite Joins Up",
  "alternate_titles": [
    "Snowhite: A Merry Xmas-New Year Revue"
  ],
  "type": "musical revue (pantomime/topical satire)",
  "venue": "Camp theatre (exact hall not specified)",
  "location": "Hay Internment Camp, New South Wales, Australia",
  "date": "1941",
  "presented_by": "Internees of Hay Internment Camp",
  "overall_credits": {
    "written_produced_directed_by": "Doc K. Sternberg",
    "music": "Ray Martin",
    "musical_arrangements_and_piano": [
      "Jonny Flynn",
      "Rudolf Laqueur",
      "Herbert Voss"
    ],
    "drums": "Kurt Mayer",
    "scenery": [
      "Emil Wittenberg",
      "Klaus Friedeberger",
      "Fritz Schoenbach",
      "Heinz Tichauer"
    ],
    "costumes": [
      "George Blank",
      "Willy Herr",
      "Kurt Rosenberg"
    ],
    "choreography": "Klaus Begach",
    "masks": "Bernhard Joseph",
    "stage_manager": "David Rummelsburg",
    "assistant_stage_manager": "H. P. Kessler",
    "technical_staff": [
   

In [7]:
key_counts = Counter()
for c in concerts:
    key_counts.update(c.keys())

print(f"Field frequency across all {len(concerts)} records:")
for key, count in key_counts.most_common():
    print(f"  {key:<30} {count}/{len(concerts)}")


Field frequency across all 26 records:
  title                          26/26
  type                           26/26
  venue                          26/26
  location                       26/26
  date                           26/26
  additional_notes               26/26
  Complete credits               26/26
  acts                           24/26
  overall_credits                15/26
  presented_by                   9/26
  performers                     6/26
  alternate_titles               4/26
  composer_lyricist              2/26
  lyrics                         2/26
  performers_list                2/26
  cast                           2/26


A handful of fields (`title`, `type`, `venue`, `location`, `date`, `additional_notes`, `Complete credits`) appear in every record — these form a reliable core. Everything else (`overall_credits`, `acts`, `performers`, `cast`, `lyrics`, ...) is present only when the source document actually had that kind of information, so any code that harvests from them needs to check for presence rather than assume it.

Note also the inconsistent naming: `Complete credits` (capitalized, space) sits alongside `snake_case` keys like `overall_credits` — a small but real reminder to `.get()` fields by their *exact* key rather than guessing a convention.

Now let's harvest the four target fields one at a time: **type**, **date**, **location**, and **persons + roles**.

### Event type

`type` is free text written by the cataloger for each document, not a controlled vocabulary — so there are as many distinct `type` strings as there are shades of "concert" in the collection. Let's look at them all before deciding how to group them.

In [8]:
raw_types = [c.get("type") for c in concerts]
for t in sorted(set(raw_types)):
    print("-", t)


- chamber concert
- comic amateur play (one-act parody melodrama)
- communal religious/cultural song evening (songbook)
- concert
- concert (chamber recital)
- concert (orchestral, with organ)
- concert (piano & vocal recital)
- full-length play
- musical revue
- musical revue (pantomime/topical satire)
- musical revue (three-act pantomime/wartime satire)
- piano recital
- play (theatrical production, likely Noel Coward's 'Hay Fever')
- shipboard variety concert with one-act play
- shipboard variety revue
- shipboard variety show
- standalone original song (handwritten manuscript)
- standalone original song (unit marching song)
- standalone original songs (handwritten manuscript)
- theatrical production with orchestra
- variety show
- vocal recital


In [9]:
#A simple keyword heuristic bucketing the free-text `type` field into the three main
#categories the workshop cares about, plus a catch-all for anything that doesn't fit.
#This is a starting point to adjust, not a definitive classification.
def classify_type(raw_type):
    t = (raw_type or "").lower()
    if "revue" in t or "variety" in t or "pantomime" in t:
        return "Revue/Variety"
    if "play" in t or "theatrical" in t or "drama" in t:
        return "Theatrical"
    if "concert" in t or "recital" in t:
        return "Concert"
    return "Other"

for raw_type in sorted(set(raw_types)):
    print(f"{classify_type(raw_type):<15} <- {raw_type}")


Concert         <- chamber concert
Theatrical      <- comic amateur play (one-act parody melodrama)
Other           <- communal religious/cultural song evening (songbook)
Concert         <- concert
Concert         <- concert (chamber recital)
Concert         <- concert (orchestral, with organ)
Concert         <- concert (piano & vocal recital)
Theatrical      <- full-length play
Revue/Variety   <- musical revue
Revue/Variety   <- musical revue (pantomime/topical satire)
Revue/Variety   <- musical revue (three-act pantomime/wartime satire)
Concert         <- piano recital
Theatrical      <- play (theatrical production, likely Noel Coward's 'Hay Fever')
Revue/Variety   <- shipboard variety concert with one-act play
Revue/Variety   <- shipboard variety revue
Revue/Variety   <- shipboard variety show
Other           <- standalone original song (handwritten manuscript)
Other           <- standalone original song (unit marching song)
Other           <- standalone original songs (handwritten 

### Date

Dates in this collection are just bare years (some events couldn't be dated more precisely than that), or the literal string `"Not specified"` when even the year is unknown. We'll parse what we can into an integer year and leave the rest as missing, rather than guessing.

In [10]:
def parse_year(date_str):
    try:
        return int(date_str)
    except (ValueError, TypeError):
        return None

years = [parse_year(c.get("date")) for c in concerts]
print(f"{sum(y is None for y in years)} of {len(years)} records have no usable year.")


6 of 26 records have no usable year.


### Location

`location` is also free text, and it mixes several levels of geography into one string — e.g. `"Hay Internment Camp, New South Wales, Australia"`. For chart axis labels we don't want the full string, so we'll take everything before the first comma as a short place label.

Watch what this reveals: `"Unspecified Internment camp"`, `"Unspecified internment camp"`, and `"An unspecified internment/military camp"` are three *different* strings for what's probably the same real-world "unknown camp" idea — a good example of why place names usually need a normalization/canonicalization pass before they become nodes in a network. We're flagging it here rather than silently fixing it, since deciding how to merge (or not merge) categories like this is exactly the kind of judgment call a workshop should make explicitly.

In [11]:
def place_short(location):
    if not location or location == "Not specified":
        return "Unknown"
    return location.split(",")[0].strip()

places = [place_short(c.get("location")) for c in concerts]
print(pd.Series(places).value_counts())


Hay Internment Camp                                 6
Tatura Internment Camp                              5
Melbourne                                           4
Unknown                                             3
Aboard His Majesty's Transport D3 (troopship)       2
Early Internment Camp                               1
Unspecified Internment camp                         1
Unspecified internment camp                         1
An unspecified internment/military camp             1
Sandwich                                            1
Aboard T.S.S. "Largs Bay" (Captain T. V. Roberts    1
Name: count, dtype: int64


In [12]:
concerts_df = pd.DataFrame([
    {
        "title": c["title"],
        "type_raw": c.get("type"),
        "type_category": classify_type(c.get("type")),
        "date_raw": c.get("date"),
        "year": parse_year(c.get("date")),
        "location": c.get("location"),
        "place_short": place_short(c.get("location")),
        "num_credited": len(c.get("Complete credits", [])),
    }
    for c in concerts
])
concerts_df.head(10)


,title,type_raw,type_category,date_raw,year,location,place_short,num_credited
0,Snowhite Joins Up,musical revue (pantomime/topical satire),Revue/Variety,1941,1941.0,"Hay Internment Camp, New South Wales, Australia",Hay Internment Camp,49
1,"Don't Release Me, Mr. Layton (song manuscript)",standalone original song (handwritten manuscript),Other,1941,1941.0,"Hay Internment Camp, New South Wales, Australia",Hay Internment Camp,0
2,Die Geschichte vom braven Soldaten Schwejk (Th...,theatrical production with orchestra,Theatrical,1941,1941.0,"Tatura Internment Camp, Victoria, Australia",Tatura Internment Camp,44
3,Wir Reisen um die Welt / We Travel Round the W...,musical revue,Revue/Variety,1941,1941.0,"Tatura Internment Camp, Victoria, Australia",Tatura Internment Camp,32
4,Grosses Unterhaltungskonzert (Grand Entertainm...,concert,Concert,1942,1942.0,"Tatura Internment Camp, Victoria, Australia",Tatura Internment Camp,12
5,Arien Abend (Aria Evening),vocal recital,Concert,1941,1941.0,"Tatura Internment Camp, Victoria, Australia",Tatura Internment Camp,2
6,M. Pietruschka Chamber Concert (untitled progr...,chamber concert,Concert,1940,1940.0,"Hay Internment Camp, New South Wales, Australia",Hay Internment Camp,10
7,1st Concert (Recreation Department),concert,Concert,1940,1940.0,Early Internment Camp,Early Internment Camp,9
8,Sergeant Snow White,musical revue (three-act pantomime/wartime sat...,Revue/Variety,1943,1943.0,"Melbourne, Victoria, Australia",Melbourne,65
9,Journey's End,full-length play,Theatrical,Not specified,NaN,Not specified,Unknown,10


## Persons and the hierarchy of roles

Person names live in *three* different places in a record, each harder to harvest than the last:

1. **`Complete credits`** — every record has this: a flat, already-cleaned list of everyone mentioned anywhere in that record (parenthetical uncertainty notes like `"(possibly ...)"` already stripped out). No role information, but reliable and always present.
2. **`overall_credits`** (and, on some records, `performers`, `performers_list`, or `cast`) — this is where the actual *hierarchy of roles* lives: a dictionary mapping a role label (e.g. `"stage_manager"`, `"cast"`, `"orchestra_members"`) to the person(s) in that role. But the value's shape varies per role: a single name (string), several names (list), or even a nested sub-grouping (a dictionary), such as pianists split out by season. Harvesting this requires handling all three shapes.
3. **Inside `acts` → `songs` → `credits`** — free-text, per-scene cast blurbs like `"Prince Charming - Eric Liffmann; The Witch/Grandma - George Blank"`. This is the messiest layer (would need regex or NLP to parse reliably) and we won't fully parse it here — it's left as an extension exercise.

Let's harvest layer 2 first, since it's where the interesting role structure is, then fall back on layer 1 (`Complete credits`) for anything downstream that needs to be reliable across *every* record.

In [13]:
def flatten_roles(value, role_path=()):
    #Walks a role -> person(s) structure and yields (role_label, person) pairs, regardless of
    #whether the value at this level is a single name, a list of names, or a nested sub-grouping.
    role_label = " / ".join(role_path)
    if isinstance(value, str):
        yield (role_label, value)
    elif isinstance(value, list):
        for item in value:
            yield from flatten_roles(item, role_path)
    elif isinstance(value, dict):
        for sub_role, sub_value in value.items():
            yield from flatten_roles(sub_value, role_path + (sub_role,))

#Demo on one record with a nested (dict-valued) role, so all three shapes are visible at once
demo = next(c for c in concerts if "overall_credits" in c and any(
    isinstance(v, dict) for v in c["overall_credits"].values()
))
print(f"Roles in: {demo['title']}\n")
for role, person in flatten_roles(demo["overall_credits"]):
    print(f"  {role:<35} {person}")


Roles in: Sergeant Snow White

  written_and_directed_by             Sergeant Doc K. Sternberg
  choreography                        Cpl. A. P. Schmitz
  ballet_and_dances_arranged_by       Cpl. A. P. Schmitz
  ballet_and_dances_arranged_by       Max Lewinsky
  musical_arrangements                Ptes. S. Cohn
  musical_arrangements                H. Fichmann
  special_musical_arrangements_by     Ray Martin
  band                                The Band of the 8th Australian Employment Company, A.M.F., conducted by L'Cpl. O. Fleischer
  at_the_pianos / April season        Ptes. E. Fraenkel
  at_the_pianos / April season        H. Fichmann
  at_the_pianos / April season        Herbert Voss
  at_the_pianos / April season        S. Cohn
  at_the_pianos / May season (expanded) L/Cpl. H. Portnoj
  at_the_pianos / May season (expanded) Ptes. E. Fraenkel
  at_the_pianos / May season (expanded) H. Fichmann
  at_the_pianos / May season (expanded) Herbert Voss
  at_the_pianos / May season (expan

In [14]:
role_rows = []
for c in concerts:
    for source_key in ("overall_credits", "performers", "performers_list", "cast"):
        block = c.get(source_key)
        if block:
            for role, person in flatten_roles(block):
                role_rows.append({"title": c["title"], "role": role, "person": person})

roles_df = pd.DataFrame(role_rows)
print(f"{len(roles_df)} (role, person) pairs harvested from {roles_df['title'].nunique()} records.\n")
roles_df["role"].value_counts().head(10)


250 (role, person) pairs harvested from 20 records.



role
cast                                     19
orchestra_members                        19
collaborators                            11
orchestra                                 9
ballet_personnel                          9
seven_dwarfs_personnel                    7
at_the_pianos / May season (expanded)     6
swing_choir_personnel                     6
music                                     6
accordionists                             6
Name: count, dtype: int64

`roles_df` is rich (it keeps the role hierarchy) but incomplete — only 15 of the 26 records have an `overall_credits`/`performers`/`cast` block at all (see the field-frequency table above), and the free-text credits buried inside `acts` aren't included. For anything downstream that needs to work across the *whole* archive reliably — like preparing the Person–Place network — we'll fall back to layer 1, `Complete credits`, even though it costs us the role information.

In [15]:
credits_long = pd.DataFrame([
    {"title": c["title"], "person": person}
    for c in concerts
    for person in c.get("Complete credits", [])
])
print(f"{len(credits_long)} (title, person) pairs across {credits_long['person'].nunique()} unique people.")
credits_long.head()


303 (title, person) pairs across 242 unique people.


,title,person
0,Snowhite Joins Up,Doc K. Sternberg
1,Snowhite Joins Up,Ray Martin
2,Snowhite Joins Up,Jonny Flynn
3,Snowhite Joins Up,Rudolf Laqueur
4,Snowhite Joins Up,Herbert Voss


## Visualizing the archive

With `concerts_df` in hand, a few quick charts with **Plotly Express** (`px`) show the distribution of events by type, place, and time. Plotly Express charts are interactive by default (hover for details, zoom, pan) — handy for exploring a small archive like this one live in a workshop.

In [16]:
fig = px.bar(
    concerts_df["type_category"].value_counts().reset_index(name="count"),
    x="type_category", y="count",
    title="Events by type",
    labels={"type_category": "Event type", "count": "Number of events"},
)
fig.show()


In [17]:
place_counts = concerts_df["place_short"].value_counts().reset_index(name="count")
fig = px.bar(
    place_counts,
    x="count", y="place_short",
    orientation="h",
    title="Events by location",
    labels={"place_short": "Location", "count": "Number of events"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()


In [18]:
by_year = concerts_df.dropna(subset=["year"]).astype({"year": int})
year_counts = by_year["year"].value_counts().sort_index().reset_index(name="count")
fig = px.bar(
    year_counts,
    x="year", y="count",
    title="Events by year",
    labels={"year": "Year", "count": "Number of events"},
)
fig.update_xaxes(type="category")
fig.show()


In [19]:
fig = px.scatter(
    by_year,
    x="year", y="place_short",
    color="type_category", size="num_credited",
    hover_name="title",
    title="Events over time, by place and type",
    labels={"year": "Year", "place_short": "Location", "type_category": "Event type", "num_credited": "People credited"},
)
fig.update_xaxes(type="category")
fig.show()


In [20]:
type_by_year = by_year.groupby(["year", "type_category"]).size().reset_index(name="count")
fig = px.bar(
    type_by_year,
    x="year", y="count", color="type_category",
    title="Event type composition by year",
    labels={"year": "Year", "count": "Number of events", "type_category": "Event type"},
)
fig.update_xaxes(type="category")
fig.show()


## Setting up for the network stage

The next step beyond this notebook is a **bi-nodal network**: two kinds of nodes (people and places), with an edge whenever a person was involved in an event at that place. That graph itself will be built with `networkx` and rendered with `pyvis`, using code shared separately — this notebook's job is just to produce a clean edge list ready to hand off.

We build it by joining `credits_long` (title → person, from `Complete credits`) against `concerts_df` (title → place), then dropping the title and de-duplicating — a person who appears in multiple songs within the same event should only produce one edge to that event's place.

In [21]:
person_place_edges = (
    credits_long
    .merge(concerts_df[["title", "place_short"]], on="title")
    .drop(columns="title")
    .drop_duplicates()
    .rename(columns={"place_short": "place"})
)

print(f"{len(person_place_edges)} person-place edges, "
      f"{person_place_edges['person'].nunique()} unique people, "
      f"{person_place_edges['place'].nunique()} unique places.")
person_place_edges.head(10)


280 person-place edges, 242 unique people, 8 unique places.


,person,place
0,Doc K. Sternberg,Hay Internment Camp
1,Ray Martin,Hay Internment Camp
2,Jonny Flynn,Hay Internment Camp
3,Rudolf Laqueur,Hay Internment Camp
4,Herbert Voss,Hay Internment Camp
5,Kurt Mayer,Hay Internment Camp
6,Emil Wittenberg,Hay Internment Camp
7,Klaus Friedeberger,Hay Internment Camp
8,Fritz Schoenbach,Hay Internment Camp
9,Heinz Tichauer,Hay Internment Camp


## A Bi-Nodal Network of People and Places

A network is just **nodes** (entities) connected by **edges** (relationships), optionally with **weights** on those edges indicating how strong a relationship is. So far every network we might build from this archive has had one *kind* of node. Here we build one with **two kinds at once**: people and places — a "bi-nodal" (or *bipartite*) network. As the general network tutorial puts it: "You could even have different kinds of nodes in the same network... distinguished by color or shape." That's exactly what we do below: people are one color, places are another.

The edges themselves come straight from `person_place_edges`-style data we already built above (`credits_long` joined to `concerts_df`) — an edge connects a person to a place whenever that person is credited on an event that happened there. This time we'll also count *how many* events link each person to each place, and use that count as the edge **weight**.

In [22]:
#Group by (person, place) and count how many distinct events link them, so a person
#credited on 3 events at the same camp gets one edge of weight 3, not three separate edges.
person_place_counts = (
    credits_long
    .merge(concerts_df[["title", "place_short"]], on="title")
    .rename(columns={"place_short": "place"})
    .groupby(["person", "place"])
    .size()
    .reset_index(name="weight")
)

print(f"{len(person_place_counts)} weighted person-place edges.")
person_place_counts.sort_values("weight", ascending=False).head(10)


280 weighted person-place edges.


,person,place,weight
134,H. W. Katz,Tatura Internment Camp,3
66,Emil Wittenberg,Hay Internment Camp,3
207,Mrs. O. Keel,Aboard His Majesty's Transport D3 (troopship),2
266,Tony Firth,Aboard His Majesty's Transport D3 (troopship),2
234,Ray Martin,Aboard His Majesty's Transport D3 (troopship),2
37,Capt. O'Gorman,Aboard His Majesty's Transport D3 (troopship),2
32,Bernard Bell,Aboard His Majesty's Transport D3 (troopship),2
249,S. Cohn,Hay Internment Camp,2
251,S. Cohn,Tatura Internment Camp,2
254,S. Lehmann,Tatura Internment Camp,2


### Building the graph with NetworkX

We add the two node types in separate calls to `G.add_nodes_from()`, each tagged with a `color` (what pyvis will actually draw), a `bipartite` flag (the NetworkX-standard way of marking a graph's two node sets — useful if you later want NetworkX's own bipartite algorithms), and a `node_type` label (handy for filtering later, e.g. `[n for n, d in G.nodes(data=True) if d["node_type"] == "place"]`).

Edges then come straight from `person_place_counts`, with the event count carried over as both the NetworkX `weight` attribute and pyvis's `width` attribute (thicker edge = more shared events). Node `size` is scaled by degree — the same *centrality* idea from the general tutorial: people or places connected to many others end up bigger and get pulled toward the center once physics is applied.

In [23]:
import networkx as nx

#Two colors for our two kinds of nodes
PERSON_COLOR = "#4C72B0"  # blue
PLACE_COLOR = "#DD8452"   # orange

G = nx.Graph()

people = person_place_counts["person"].unique()
places = person_place_counts["place"].unique()

G.add_nodes_from(people, bipartite=0, node_type="person", color=PERSON_COLOR)
G.add_nodes_from(places, bipartite=1, node_type="place", color=PLACE_COLOR, shape="square")

for _, row in person_place_counts.iterrows():
    G.add_edge(row["person"], row["place"], weight=int(row["weight"]))

for _, _, edge_data in G.edges(data=True):
    edge_data["width"] = edge_data["weight"]

#Scale node size and add a hover label based on degree (centrality)
for node, node_data in G.nodes(data=True):
    degree = G.degree(node)
    node_data["size"] = 10 + 3 * degree
    node_data["title"] = f"{node} ({node_data['node_type']}, {degree} connection{'s' if degree != 1 else ''})"

print(f"{G.number_of_nodes()} nodes ({len(people)} people, {len(places)} places), "
      f"{G.number_of_edges()} edges. Bipartite: {nx.is_bipartite(G)}")


250 nodes (242 people, 8 places), 280 edges. Bipartite: True


### Rendering with Pyvis

With ~250 nodes, the default physics leaves everything clumped in the middle — illegible. Setting `solver: forceAtlas2Based` (the same option used in the general tutorial) spreads the graph out so clusters of people around a shared place become visible, and lets you drag nodes around interactively once rendered.

`cdn_resources="in_line"` bundles Pyvis's JS/CSS directly into the output HTML — slightly larger file, but it means the graph still renders with no internet connection and doesn't hit the Chrome/Safari display issues Pyvis warns about under the default `"local"` setting.

In [24]:
from pyvis import network as net

person_place_network = net.Network(
    notebook=True,
    width="900px",
    height="900px",
    bgcolor="#111111",
    font_color="white",
    cdn_resources="in_line",
)

person_place_network.set_options("""
{
  "physics": {
    "enabled": true,
    "forceAtlas2Based": {
      "springLength": 100
    },
    "solver": "forceAtlas2Based"
  }
}
""")

person_place_network.from_nx(G)
person_place_network.show("person_place_network.html")


person_place_network.html


### Optional: Louvain Community Detection

Since people here only connect to places (never to each other directly), Louvain communities will mostly just rediscover "people tied to the same place" — but it's worth running anyway, because it's precisely the people credited at *more than one* place (e.g. someone who moved from Hay to Tatura to Melbourne, following the pattern the archive overview describes) who end up as bridges between communities, visually pulling two place-clusters together. Those bridge figures are often the most historically interesting nodes in the whole graph.

This cell needs the `python-louvain` package (`pip install python-louvain`, imported as `community`) — it's optional, so the cell checks for it rather than failing the whole notebook if it isn't installed.

In [25]:
try:
    from community import community_louvain
    from copy import deepcopy

    def add_communities(graph):
        graph = deepcopy(graph)
        partition = community_louvain.best_partition(graph)
        nx.set_node_attributes(graph, partition, "group")
        return graph

    G_communities = add_communities(G)
    print(f"{len(set(nx.get_node_attributes(G_communities, 'group').values()))} communities detected.")

    community_network = net.Network(
        notebook=True, width="900px", height="900px",
        bgcolor="#111111", font_color="white", cdn_resources="in_line",
    )
    community_network.set_options("""
    {
      "physics": {
        "enabled": true,
        "forceAtlas2Based": {"springLength": 100},
        "solver": "forceAtlas2Based"
      }
    }
    """)
    community_network.from_nx(G_communities)
    community_network.show("person_place_network_louvain.html")

except ImportError:
    print("python-louvain isn't installed. Run `pip install python-louvain` and re-run this cell to see communities.")


7 communities detected.
person_place_network_louvain.html
